# V 표현·N 전파·전체 CLV 손실 M5: Dunnhumby seed 42

동일 실행에서 두 arm만 처음부터 학습합니다. 기준 arm은 `q_V 가격위치 표현 + 현재 M4`이며, 후보 arm은 같은 구조에 `q_N` 기반 M3 첫 전파 재배분만 추가합니다. M3는 반복 구매한 과거 상품을 강화하지 않습니다. 반복구매 빈도가 높은 사용자일수록 반복횟수가 큰 과거 상품의 전파 몫을 줄이고 덜 반복한 과거 관계에 같은 총량을 재배분합니다.

이 실행은 `historical_development_days_684_690`의 seed 42 방향성 확인입니다. 후보가 두 Top-10 경제지표에서 기준을 모두 이길 때만 N-free 관계 대조군과 degree-matched N 순열을 다음 단계에서 추가합니다. 이번 결과만으로 CLV 귀속·유의성·안정성·일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'd5d7462ec459a6e67d160ed4b853afb7b0ebfcb3'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_value_basis_repeat_frequency_first_hop_screen import (
    MODEL_IDS,
    configure_value_basis_repeat_frequency_screen,
    preflight_summary,
    run_value_basis_repeat_frequency_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_value_basis_repeat_frequency_screen(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m5_value_basis_repeat_frequency_'
        'first_hop_two_arm_development_screen_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == list(MODEL_IDS)
assert summary['reused_models'] == []
assert summary['prior_result_file_required'] is False
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['train_pairs_excluded_from_evaluation'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['binary_base_graph'] is True
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['one_training_loop_and_optimizer_per_arm'] is True
assert summary['fixed']['pretraining_or_freezing'] is False
assert summary['fixed']['external_reranking'] is False
assert summary['m2']['input'].startswith('observed q_V only')
assert summary['m3']['per_user_first_hop_mass_preserved'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_value_basis_repeat_frequency_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) M3-off 기준과 N 반복편향 보정 M5 절대지표')
show(result_df)
print('2) N-M3 증분 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) ID·V 점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) M3 반복횟수 관계·전파 진단')
print(json.dumps(result_df.attrs['m3_repeat_frequency_diagnostics'], ensure_ascii=False, indent=2))
print('5) 학습 전 작동·동일성 점검')
print(json.dumps(result_df.attrs['operational_preflight'], ensure_ascii=False, indent=2))
print('6) Top-10 변경 진단')
print(json.dumps(result_df.attrs['ranking_change'], ensure_ascii=False, indent=2))
print('7) 사전 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('8) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))